# 💰 Sales Data – Handling Missing Values with Pandas

---

**📂 Dataset:** `sales.csv`

**🔑 Concepts covered:**
- Dropping rows with missing values using `.dropna()`
- Filling missing values using `.fillna()`
- Strategies for handling NaN in sales figures
- Exporting the cleaned data to Excel

**🎯 What you will learn:**  
You will work through a sales dataset to understand when to drop versus fill missing data, and how to deliver a clean Excel file to stakeholders.

---


## Step 1: Import Pandas

**What this does:** We start by importing the **Pandas** library and giving it the short alias `pd`. Pandas is the go-to Python library for working with tabular data (think: Excel, but in code).

**Key concept:** `import pandas as pd` — loads the library so we can use `pd.DataFrame`, `pd.read_csv()`, etc.


In [41]:
import pandas as pd

## Step 2: Load the Dataset

**What this does:** We read a CSV file from disk into a Pandas **DataFrame** – a table with rows and columns. This is almost always the very first step in a data-analysis project.

**Key concept:** `pd.read_csv()` — reads a comma-separated file and returns a DataFrame you can query and manipulate.


In [42]:
df = pd.read_csv("sales.csv")

## Step 3: Inspect Data Types and Non-Null Counts

**What this does:** We call `.info()` to get a concise summary: column names, non-null counts, and data types. This is the fastest way to spot missing values and type mismatches.

**Key concept:** `.info()` — prints column names, number of non-null entries, and dtype for each column.


In [43]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


## Step 4: Count Missing Values

**What this does:** We count how many **null / NaN** values exist in each column. Missing data is one of the most common data-quality problems and must be addressed before analysis.

**Key concept:** `.isnull().sum()` — `.isnull()` creates a boolean mask (True where data is missing); `.sum()` counts the Trues per column.


In [44]:
df.isna().sum().sum()

np.int64(6826)

## Step 5: Drop Rows/Columns with Missing Values

**What this does:** We **remove** rows (or columns) that contain missing values. Use this when the missing data is too sparse to impute reliably.

**Key concept:** `.dropna()` — by default drops any row with at least one NaN. Use `axis=1` to drop columns instead.


In [45]:
df.dropna(inplace=True)

## Step 6: Code Block

**What this does:** The cell below runs the following Python code:

```python
df
```

Read the code carefully – each line either transforms the data or displays a result.


In [46]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9984,TXN_3142496,Smoothie,UNKNOWN,4.0,4.0,Cash,Takeaway,2023-07-27
9986,TXN_2858441,Sandwich,2,4.0,8.0,Credit Card,In-store,2023-12-14
9991,TXN_3897619,Sandwich,3,4.0,12.0,Cash,Takeaway,2023-02-24
9992,TXN_2739140,Smoothie,4,4.0,16.0,UNKNOWN,In-store,2023-07-05


## Step 7: Count Missing Values

**What this does:** We count how many **null / NaN** values exist in each column. Missing data is one of the most common data-quality problems and must be addressed before analysis.

**Key concept:** `.isnull().sum()` — `.isnull()` creates a boolean mask (True where data is missing); `.sum()` counts the Trues per column.


In [47]:
df.isna().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

## Step 8: Detect Duplicate Rows

**What this does:** We check whether any rows appear more than once in the dataset. Duplicates can skew aggregations and model training.

**Key concept:** `.duplicated()` — returns a boolean Series; `True` for every row that is an exact copy of a previous row.


In [48]:
df.duplicated().sum()

np.int64(0)

## Step 9: Count Unique Values

**What this does:** We count how many times each **unique value** appears in a column. This is perfect for exploring categorical data.

**Key concept:** `.value_counts()` — returns a Series sorted by frequency, highest first.


In [49]:
pd.Series(df['Transaction ID'].unique()).str.startswith("TXN").value_counts()

True    4550
Name: count, dtype: int64

### as we applu df["column_name].unique() -> we find that each column has some valuse as UNKNOWN and ERROR now we have to handle these missing values

The difference lies in what is being filtered: **row-filtering (dropping entire rows)** vs. **element-wise masking (replacing matching cells with `NaN`)**.

---

### 1. `df[~df.isin(["UNKNOWN", "ERROR"]).any(axis=1)]`

* **What it does:** **Filters and drops entire rows.**
* **How it works:**
1. `df.isin(...)` generates a boolean DataFrame of the same shape as `df`.
2. `.any(axis=1)` checks across columns for each row, returning a 1D Series of `True` (if the row has *at least one* `"UNKNOWN"` or `"ERROR"`) and `False`.
3. `~` inverts that Series (`True` becomes `False`).
4. Passing this 1D boolean Series into `df[...]` selects only the rows where **none** of those target values exist anywhere in that row.


* **Output Shape:** Same number of columns, but fewer (or equal) rows.
* **Cell Values:** Values remain completely unchanged.

---

### 2. `df[~df.isin(["UNKNOWN", "ERROR"])]`

* **What it does:** **Element-wise masking.**
* **How it works:**
1. `~df.isin(...)` produces a **2D boolean DataFrame** where cells matching `"UNKNOWN"` or `"ERROR"` are `False`, and all other cells are `True`.
2. When you pass a 2D boolean DataFrame of matching shape into `df[...]`, pandas keeps the cells that are `True` and turns any cell that is `False` into **`NaN`**.


* **Output Shape:** Exactly the same dimensions (rows and columns) as the original DataFrame.
* **Cell Values:** Any occurrence of `"UNKNOWN"` or `"ERROR"` is replaced by `NaN`. No rows are dropped.

---

### Quick Comparison

| Feature | `df[~df.isin(...).any(axis=1)]` | `df[~df.isin(...)]` |
| --- | --- | --- |
| **Action** | Drops matching rows entirely | Keeps all rows; masks matching cells |
| **Target Values Become** | Removed along with the row | Replaced with `NaN` |
| **Output Shape** | `(fewer_rows, same_cols)` | `(same_rows, same_cols)` |
| **Input to `df[...]**` | 1D Series (`[True, False, ...]`) | 2D DataFrame of booleans |

## Step 10: Code Block

**What this does:** The cell below runs the following Python code:

```python
df.isin(["UNKNOWN", "ERROR"]).sum()
```

Read the code carefully – each line either transforms the data or displays a result.


In [50]:
df.isin(["UNKNOWN", "ERROR"]).sum()

Transaction ID        0
Item                295
Quantity            146
Price Per Unit      168
Total Spent         155
Payment Method      346
Location            468
Transaction Date    128
dtype: int64

## Step 11: Code Block

**What this does:** The cell below runs the following Python code:

```python
df.isin(["UNKNOWN"]).sum()
```

Read the code carefully – each line either transforms the data or displays a result.


In [51]:
df.isin(["UNKNOWN"]).sum()

Transaction ID        0
Item                159
Quantity             75
Price Per Unit       80
Total Spent          75
Payment Method      167
Location            229
Transaction Date     60
dtype: int64

## Step 12: Code Block

**What this does:** The cell below runs the following Python code:

```python
(df=="ERROR").sum()
```

Read the code carefully – each line either transforms the data or displays a result.


In [52]:
(df=="ERROR").sum()

Transaction ID        0
Item                136
Quantity             71
Price Per Unit       88
Total Spent          80
Payment Method      179
Location            239
Transaction Date     68
dtype: int64

## Step 13: Code Block

**What this does:** The cell below runs the following Python code:

```python
df = df[~df.isin(["UNKNOWN","ERROR"]).any(axis=1)]
```

Read the code carefully – each line either transforms the data or displays a result.


In [53]:
df = df[~df.isin(["UNKNOWN","ERROR"]).any(axis=1)]

## Step 14: Code Block

**What this does:** The cell below runs the following Python code:

```python
df
```

Read the code carefully – each line either transforms the data or displays a result.


In [54]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
10,TXN_2548360,Salad,5,5.0,25.0,Cash,Takeaway,2023-11-07
12,TXN_7619095,Sandwich,2,4.0,8.0,Cash,In-store,2023-05-03
...,...,...,...,...,...,...,...,...
9975,TXN_9668108,Cake,1,3.0,3.0,Cash,In-store,2023-01-20
9979,TXN_9933628,Smoothie,5,4.0,20.0,Cash,In-store,2023-07-20
9986,TXN_2858441,Sandwich,2,4.0,8.0,Credit Card,In-store,2023-12-14
9991,TXN_3897619,Sandwich,3,4.0,12.0,Cash,Takeaway,2023-02-24


## Step 15: Check Data Types

**What this does:** We inspect the **data type** of every column. Wrong types (e.g. a date stored as a string) are a very common data-quality issue.

**Key concept:** `.dtypes` — returns a Series mapping each column name to its dtype (`int64`, `object`, `float64`, etc.).


In [55]:
df.dtypes

Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str
dtype: object

## Step 16: Convert Data Types

**What this does:** We **cast** a column to a different data type (e.g. string to integer, object to datetime). Correct types unlock the right operations – you can't do arithmetic on a string column.

**Key concept:** `.astype(dtype)` — returns a new Series with the values converted to `dtype`.


In [56]:
df['Quantity'] = df["Quantity"].astype(int)

## Step 17: Check Data Types

**What this does:** We inspect the **data type** of every column. Wrong types (e.g. a date stored as a string) are a very common data-quality issue.

**Key concept:** `.dtypes` — returns a Series mapping each column name to its dtype (`int64`, `object`, `float64`, etc.).


In [57]:
df.dtypes

Transaction ID        str
Item                  str
Quantity            int64
Price Per Unit        str
Total Spent           str
Payment Method        str
Location              str
Transaction Date      str
dtype: object

## Step 18: Parse Dates

**What this does:** We convert a column of date strings into Pandas **datetime** objects. Once parsed as datetimes, we can sort, filter, and extract year/month/day easily.

**Key concept:** `pd.to_datetime(col)` — intelligently parses many date formats into `datetime64` dtype.


In [58]:
df['Transaction Date'] = pd.to_datetime(df["Transaction Date"])

## Step 19: Check Data Types

**What this does:** We inspect the **data type** of every column. Wrong types (e.g. a date stored as a string) are a very common data-quality issue.

**Key concept:** `.dtypes` — returns a Series mapping each column name to its dtype (`int64`, `object`, `float64`, etc.).


In [59]:
df.dtypes

Transaction ID                 str
Item                           str
Quantity                     int64
Price Per Unit                 str
Total Spent                    str
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object

## Step 20: Sort the DataFrame

**What this does:** We **sort** rows by one or more columns.

**Key concept:** `.sort_values(by='col')` — sorts ascending by default; pass `ascending=False` to reverse.


In [62]:
df = df.sort_values("Transaction Date")

## Step 21: Code Block

**What this does:** The cell below runs the following Python code:

```python
df
```

Read the code carefully – each line either transforms the data or displays a result.


In [63]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
8885,TXN_1581562,Coffee,2,2.0,4.0,Cash,In-store,2023-01-01
2726,TXN_2024598,Sandwich,1,4.0,4.0,Digital Wallet,In-store,2023-01-01
9356,TXN_2104473,Cake,3,3.0,9.0,Digital Wallet,Takeaway,2023-01-01
1806,TXN_2192787,Sandwich,5,4.0,20.0,Cash,In-store,2023-01-01
9063,TXN_9161256,Smoothie,2,4.0,8.0,Digital Wallet,In-store,2023-01-01
...,...,...,...,...,...,...,...,...
4017,TXN_6541110,Cookie,3,1.0,3.0,Cash,In-store,2023-12-30
2528,TXN_4592205,Cookie,3,1.0,3.0,Credit Card,In-store,2023-12-30
6515,TXN_7108012,Tea,1,1.5,1.5,Digital Wallet,Takeaway,2023-12-31
7441,TXN_1714348,Sandwich,2,4.0,8.0,Cash,Takeaway,2023-12-31


## Step 22: Code Block

**What this does:** The cell below runs the following Python code:

```python
pip install openpyxl
```

Read the code carefully – each line either transforms the data or displays a result.


In [66]:
pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]

[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 23: Export to Excel

**What this does:** We **save** the cleaned DataFrame as an Excel file (`.xlsx`).

**Key concept:** `.to_excel('filename.xlsx', index=False)` — requires the `openpyxl` package; produces a file you can open directly in Excel.


In [67]:
df.to_excel("sales.xlsx")

## Step 24: Load an Excel File

**What this does:** We load data from an Excel spreadsheet into a Pandas DataFrame.

**Key concept:** `pd.read_excel()` — reads one or more sheets from an `.xlsx` or `.xls` file.


In [68]:
pd.read_excel("sales.xlsx")

,Unnamed: 0,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,8885,TXN_1581562,Coffee,2,2.0,4.0,Cash,In-store,2023-01-01
1,2726,TXN_2024598,Sandwich,1,4.0,4.0,Digital Wallet,In-store,2023-01-01
2,9356,TXN_2104473,Cake,3,3.0,9.0,Digital Wallet,Takeaway,2023-01-01
3,1806,TXN_2192787,Sandwich,5,4.0,20.0,Cash,In-store,2023-01-01
4,9063,TXN_9161256,Smoothie,2,4.0,8.0,Digital Wallet,In-store,2023-01-01
...,...,...,...,...,...,...,...,...,...
3084,4017,TXN_6541110,Cookie,3,1.0,3.0,Cash,In-store,2023-12-30
3085,2528,TXN_4592205,Cookie,3,1.0,3.0,Credit Card,In-store,2023-12-30
3086,6515,TXN_7108012,Tea,1,1.5,1.5,Digital Wallet,Takeaway,2023-12-31
3087,7441,TXN_1714348,Sandwich,2,4.0,8.0,Cash,Takeaway,2023-12-31
